In [0]:
%pip install psycopg2

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import psycopg2

print(psycopg2.__version__)

2.9.13 (dt dec pq3 ext lo64)


In [0]:
from pyspark.sql import functions as F

# Conexão com o banco de dados PostgreSQL

PG_HOST = "pg-3117a07b-fixa-9a44.e.aivencloud.com"
PG_PORT = "15976"
PG_DATABASE = "db2anoPrincipal"
PG_SCHEMA = "public"

PG_USER = dbutils.secrets.get(
    scope="fixa",
    key="postgres-user"
)

PG_PASSWORD = dbutils.secrets.get(
    scope="fixa",
    key="postgres-password"
)

JDBC_URL = (
    f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
    "?sslmode=require"
)


In [0]:
def ler_postgres(nome_objeto):
    """
    Lê uma tabela ou view do PostgreSQL via JDBC.
    """
    objeto_completo = f"{PG_SCHEMA}.{nome_objeto}"

    return (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", objeto_completo)
        .option("user", PG_USER)
        .option("password", PG_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .option("fetchsize", "10000")
        .load()
    )

In [0]:
def adicionar_auditoria(df, nome_objeto):
    """
    Adiciona metadados técnicos da ingestão.
    """
    return (
        df
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_system",
            F.lit("postgresql")
        )
        .withColumn(
            "_source_object",
            F.lit(nome_objeto)
        )
    )

In [0]:
# Objetos necessários para as camadas Silver e Gold
OBJETOS = [
    "dim_categoria_equipamento",
    "dim_categoria_problema",
    "dim_instituicao",
    "dim_local",
    "dim_status_ordem_servico",
    "dim_status_problema",
    "dim_tecnico",
    "dim_tempo",
    "dim_unidade",
    "fato_ocorrencia",
    "fato_ordem_servico",
    "fato_problema",
    "tarefa"
]

In [0]:
# Carrega os objetos da origem e adiciona auditoria
bronze_dfs = {}

for objeto in OBJETOS:
    df = ler_postgres(objeto)

    bronze_dfs[objeto] = adicionar_auditoria(
        df,
        objeto
    )

In [0]:
# Validação da quantidade de registros carregados
for nome, df in bronze_dfs.items():
    print(f"{nome}: {len(df)} registros")

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-4604897580212003>, line 3
      1 # Validação da quantidade de registros carregados
      2 for nome, df in bronze_dfs.items():
----> 3     print(f"{nome}: {len(df)} registros")

TypeError: object of type 'DataFrame' has no len()

In [0]:
# Conferência da fato principal de ordens de serviço
display(
    bronze_dfs["fato_ordem_servico"].limit(10)
)

ordem_servico_id,problema_id,tecnico_id,status_id,categoria_problema_id,unidade_id,local_id,data_criacao_id,data_prevista_id,data_inicio_id,data_conclusao_id,data_criacao,data_prevista,data_inicio,data_conclusao,prioridade,qtd_tarefas,qtd_reaberturas,qtd_reatribuicoes,tempo_pausado_minutos,tempo_espera_minutos,tempo_resolucao_minutos,esta_aberta,esta_atrasada,finalizada,foi_finalizada,concluida_no_prazo,foi_reaberta,foi_reatribuida,prioridade_nome,_ingestion_timestamp,_source_system,_source_object
41,41,16,3,1,3,41,20261026,20261107,20261026,null,2026-10-26T03:48:16.782Z,2026-11-07T03:48:16.782Z,2026-10-26T08:48:16.782Z,null,1,3,0,1,0E-18,300.000000000000000000,null,true,false,false,false,false,false,true,Média,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
42,42,16,3,2,3,42,20261025,20261106,20261025,null,2026-10-25T03:48:16.782Z,2026-11-06T03:48:16.782Z,2026-10-25T09:48:16.782Z,null,2,3,0,1,0E-18,360.000000000000000000,null,true,false,false,false,false,false,true,Baixa,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
43,43,16,3,3,3,43,20261024,20261105,20261024,null,2026-10-24T03:48:16.782Z,2026-11-05T03:48:16.782Z,2026-10-24T10:48:16.782Z,null,0,3,0,1,0E-18,420.000000000000000000,null,true,false,false,false,false,false,true,Alta,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
44,44,16,3,4,3,44,20261023,20261104,20261023,null,2026-10-23T03:48:16.782Z,2026-11-04T03:48:16.782Z,2026-10-23T11:48:16.782Z,null,1,3,0,1,0E-18,480.000000000000000000,null,true,false,false,false,false,false,true,Média,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
45,45,16,3,5,3,45,20261022,20261103,20261022,null,2026-10-22T03:48:16.782Z,2026-11-03T03:48:16.782Z,2026-10-22T12:48:16.782Z,null,2,3,0,1,0E-18,540.000000000000000000,null,true,false,false,false,false,false,true,Baixa,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
46,46,16,3,6,3,46,20261021,20261102,20261021,null,2026-10-21T03:48:16.782Z,2026-11-02T03:48:16.782Z,2026-10-21T13:48:16.782Z,null,0,3,0,1,0E-18,600.000000000000000000,null,true,false,false,false,false,false,true,Alta,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
47,47,16,3,7,3,47,20261020,20261101,20261020,null,2026-10-20T03:48:16.782Z,2026-11-01T03:48:16.782Z,2026-10-20T14:48:16.782Z,null,1,3,0,1,0E-18,660.000000000000000000,null,true,false,false,false,false,false,true,Média,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
48,48,16,3,8,3,48,20261019,20261031,20261019,null,2026-10-19T03:48:16.782Z,2026-10-31T03:48:16.782Z,2026-10-19T07:48:16.782Z,null,2,3,0,1,0E-18,240.000000000000000000,null,true,false,false,false,false,false,true,Baixa,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
49,49,16,3,9,3,49,20261018,20261030,20261018,null,2026-10-18T03:48:16.782Z,2026-10-30T03:48:16.782Z,2026-10-18T08:48:16.782Z,null,0,3,0,1,0E-18,300.000000000000000000,null,true,false,false,false,false,false,true,Alta,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico
50,50,16,3,10,3,50,20261017,20261029,20261017,null,2026-10-17T03:48:16.782Z,2026-10-29T03:48:16.782Z,2026-10-17T09:48:16.782Z,null,1,3,0,1,0E-18,360.000000000000000000,null,true,true,false,false,false,false,true,Média,2026-10-29T15:45:50.334Z,postgresql,fato_ordem_servico


In [0]:
# Conferência do schema recebido da fonte
bronze_dfs[
    "fato_ordem_servico"
].printSchema()

root
 |-- ordem_servico_id: long (nullable = true)
 |-- problema_id: long (nullable = true)
 |-- tecnico_id: long (nullable = true)
 |-- status_id: integer (nullable = true)
 |-- categoria_problema_id: integer (nullable = true)
 |-- unidade_id: long (nullable = true)
 |-- local_id: long (nullable = true)
 |-- data_criacao_id: integer (nullable = true)
 |-- data_prevista_id: integer (nullable = true)
 |-- data_inicio_id: integer (nullable = true)
 |-- data_conclusao_id: integer (nullable = true)
 |-- data_criacao: timestamp (nullable = true)
 |-- data_prevista: timestamp (nullable = true)
 |-- data_inicio: timestamp (nullable = true)
 |-- data_conclusao: timestamp (nullable = true)
 |-- prioridade: integer (nullable = true)
 |-- qtd_tarefas: integer (nullable = true)
 |-- qtd_reaberturas: integer (nullable = true)
 |-- qtd_reatribuicoes: integer (nullable = true)
 |-- tempo_pausado_minutos: decimal(38,18) (nullable = true)
 |-- tempo_espera_minutos: decimal(38,18) (nullable = true)
 |--

In [0]:
# Conferência da fato de solicitações
display(
    bronze_dfs["fato_problema"].limit(10)
)

problema_id,solicitante_id,unidade_id,local_id,categoria_equipamento_id,status_problema_id,data_criacao_id,data_criacao,quantidade_problema,esta_pendente,esta_aprovado,esta_reprovado,gerou_ordem_servico,_ingestion_timestamp,_source_system,_source_object
241,55,1,3,10,4,20261027,2026-10-27T04:35:13.926Z,1,false,false,false,true,2026-10-29T15:45:52.745Z,postgresql,fato_problema
161,54,9,161,1,0,20260727,2026-07-27T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
162,53,9,162,2,0,20260720,2026-07-20T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
163,54,9,163,3,0,20260713,2026-07-13T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
164,53,9,164,4,0,20260706,2026-07-06T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
165,54,9,165,5,0,20260629,2026-06-29T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
166,53,9,166,6,0,20260622,2026-06-22T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
167,54,9,167,7,0,20260615,2026-06-15T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
168,53,9,168,8,0,20260613,2026-06-13T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema
169,54,9,169,9,0,20260614,2026-06-14T03:48:16.782Z,1,true,false,false,false,2026-10-29T15:45:52.745Z,postgresql,fato_problema


In [0]:
# Conferência da fato de ocorrências
display(
    bronze_dfs["fato_ocorrencia"].limit(10)
)

ocorrencia_id,tecnico_id,unidade_id,local_id,categoria_problema_id,categoria_equipamento_id,data_criacao_id,data_criacao,prioridade,esta_ativo,quantidade_ocorrencia,prioridade_nome,_ingestion_timestamp,_source_system,_source_object
1,4,1,1,1,1,20261019,2026-10-19T03:48:16.782Z,0,true,1,Alta,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
2,3,1,2,2,2,20261016,2026-10-16T03:48:16.782Z,1,true,1,Média,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
3,4,1,3,3,3,20261013,2026-10-13T03:48:16.782Z,2,true,1,Baixa,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
4,3,1,4,4,4,20261010,2026-10-10T03:48:16.782Z,0,true,1,Alta,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
5,4,1,5,5,5,20261007,2026-10-07T03:48:16.782Z,1,true,1,Média,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
6,3,1,6,6,6,20261004,2026-10-04T03:48:16.782Z,2,true,1,Baixa,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
7,4,1,7,7,7,20261001,2026-10-01T03:48:16.782Z,0,true,1,Alta,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
8,3,1,8,8,8,20260928,2026-09-28T03:48:16.782Z,1,true,1,Média,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
9,4,1,9,9,9,20260925,2026-09-25T03:48:16.782Z,2,true,1,Baixa,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia
10,3,1,10,10,10,20260922,2026-09-22T03:48:16.782Z,0,true,1,Alta,2026-10-29T15:45:54.858Z,postgresql,fato_ocorrencia


In [0]:
# Conferência da tabela de tarefas usada no KPI OS-15
display(
    bronze_dfs["tarefa"].limit(10)
)

id,ordem_servico_id,status_ordem_servico,titulo,descricao,data_criacao,data_atualizacao,esta_ativo,_ingestion_timestamp,_source_system,_source_object
1,1,0,Diagnosticar problema,Diagnosticar: Computador desliga durante o uso; verificar fonte e alimentação elétrica. Ambiente: Sala de Matemática — Centro,2026-10-26T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
4,2,0,Diagnosticar problema,Diagnosticar: Condicionador apresenta gotejamento; verificar dreno e escoamento. Ambiente: Sala de Língua Portuguesa — Centro,2026-10-25T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
7,3,0,Diagnosticar problema,Diagnosticar: Luminária apresenta vibração na fixação; verificar suporte antes de liberar o uso. Ambiente: Sala de História e Geografia — Centro,2026-10-24T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
10,4,0,Diagnosticar problema,Diagnosticar: Bomba não responde ao comando do painel; verificar acionamento e controle. Ambiente: Sala de Língua Inglesa — Centro,2026-10-23T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
13,5,0,Diagnosticar problema,Diagnosticar: Suporte da câmera está solto e desloca o enquadramento; revisar fixação. Ambiente: Sala de Artes — Centro,2026-10-22T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
16,6,0,Diagnosticar problema,Diagnosticar: Roteador aquece e interrompe a conexão; verificar ventilação e dissipação. Ambiente: Laboratório de Ciências — Centro,2026-10-21T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
19,7,0,Diagnosticar problema,Diagnosticar: Ponto de rede junto à mesa não estabelece conexão; testar cabo e tomada. Ambiente: Laboratório de Informática — Centro,2026-10-20T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
22,8,0,Diagnosticar problema,Diagnosticar: Fechamento do quadro elétrico está danificado; restringir acesso e revisar trava. Ambiente: Laboratório Maker — Centro,2026-10-19T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
25,9,0,Diagnosticar problema,Diagnosticar: Indicador luminoso do bebedouro não acende; verificar sinalização e alimentação. Ambiente: Sala de Música — Centro,2026-10-18T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa
28,10,0,Diagnosticar problema,Diagnosticar: Motor acumula resíduos na carenagem; programar limpeza técnica com isolamento. Ambiente: Sala de Apoio Pedagógico — Centro,2026-10-17T04:48:16.782Z,2026-10-27T03:48:16.782Z,true,2026-10-29T15:45:56.986Z,postgresql,tarefa


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

DataFrame[]

In [0]:
# Persiste os dados brutos em Delta para consumo da Silver

for nome, df in bronze_dfs.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"workspace.bronze.{nome}"
        )
    )

    print(f"[OK] workspace.bronze.{nome}")